# disease-prediction-ml — GPU training (PhysioNet 2019)

Runs on Kaggle with **GPU + Internet** enabled. Clones the repo, installs it, prepares the attached dataset, trains, evaluates, and writes results into `artifacts/` (saved as the kernel Output).

In [ ]:
!nvidia-smi -L || echo 'no GPU'

In [ ]:
REPO = 'disease-prediction-ml'
import os, sys, subprocess
if not os.path.isdir(f'/kaggle/working/{REPO}'):
    subprocess.run(['git','clone','--depth','1',
        'https://github.com/sara-tavakoli/'+REPO+'.git'], cwd='/kaggle/working', check=True)
os.chdir(f'/kaggle/working/{REPO}')
pip = [sys.executable,'-m','pip','install','-q']
subprocess.run(pip+['--no-deps','-e','.'], check=True)
# Kaggle's P100 is sm_60; its bundled torch dropped sm_60 kernels -> install a build that keeps them
subprocess.run(pip+['torch==2.4.1','torchvision==0.19.1',
    '--index-url','https://download.pytorch.org/whl/cu121'], check=True)
subprocess.run(pip+['lightgbm>=4.0','xgboost>=2.0','shap>=0.44','mlflow>=2.9','pyyaml>=6.0','tqdm>=4.66'], check=True)
SRC = os.path.abspath('src')
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH','')
sys.path.insert(0, SRC)
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
assert (torch.zeros(2, device='cuda') + 1).sum().item() == 2, 'CUDA kernel smoke test failed'
import sepsis; print('sepsis', getattr(sepsis,'__version__','ok'))


## 1 · Prepare dataset

In [ ]:
# --- copy PhysioNet psv into a WRITABLE staging dir (input is read-only; the loader writes) ---
import subprocess, shutil, pathlib
INP = pathlib.Path('/kaggle/input')
print('input:', [p.name for p in INP.iterdir()] if INP.exists() else 'none')
setA = next((p for p in INP.rglob('training_setA') if any(p.rglob('*.psv'))), None) if INP.exists() else None
if setA is None:
    dl = pathlib.Path('/kaggle/tmp/sep'); dl.mkdir(parents=True, exist_ok=True)
    subprocess.run(['kaggle','datasets','download','-d','salikhussaini49/prediction-of-sepsis',
                    '-p',str(dl),'--unzip'], check=True)
    setA = next(p for p in dl.rglob('training_setA') if any(p.rglob('*.psv')))
src_root = setA.parent
STAGE = pathlib.Path('data/physionet')
for s in ('training_setA','training_setB'):
    d = src_root/s; inner = d/'training'
    tgt = inner if inner.is_dir() and any(inner.glob('*.psv')) else d
    out = STAGE/s; out.mkdir(parents=True, exist_ok=True)
    n = 0
    for p in tgt.glob('*.psv'):
        q = out/p.name
        if not q.exists(): shutil.copy2(p, q); n += 1
    print(s, '<-', tgt, '|', len(list(out.glob('*.psv'))), 'stays (', n, 'copied )')
ROOT = STAGE.resolve(); print('staged root:', ROOT)


## 2 · Train

In [ ]:
import subprocess
for m in ['lightgbm', 'lstm', 'gru', 'tcn', 'transformer']:
    print('=' * 20, m, '=' * 20)
    subprocess.run(['sepsis', 'train', '--config', 'configs/base.yaml',
        f'configs/model_{m}.yaml', '--set', 'data.source=physionet',
        f'data.root={ROOT}', 'data.group_by_hospital=true',
        'train.epochs=25', 'train.seed=20190804'], check=True)

In [ ]:
import pathlib
ck = list(pathlib.Path('artifacts').rglob('best.ckpt')) + list(pathlib.Path('artifacts').rglob('*.json'))
assert any(pathlib.Path('artifacts').rglob('best.ckpt')) or any(pathlib.Path('artifacts').rglob('results.json')), \
    'training produced no checkpoint/results - see the log above'
print('train artifacts OK:', [str(p) for p in ck[:6]])


## 3 · Evaluate

In [ ]:
!python scripts/update_results.py

## 4 · Show results

In [ ]:
import pathlib, IPython.display as D
for md in sorted(pathlib.Path('.').rglob('RESULTS.md')):
    D.display(D.Markdown(md.read_text()))
for png in sorted(pathlib.Path('artifacts').rglob('*.png'))[:16]:
    print(png); D.display(D.Image(str(png)))